# Titanic — v4: Interaction Features

**Version 4.** Tested five weak-with-strong feature merges; the winner was **Fare per person** (`Fare / FamilySize`) — a ratio, the one construction tree models can't build from splits. Categorical crosses (Sex×Class, Embarked×Class) helped only the linear model, and adding all merges at once hurt (overfitting). v4 = the full v1 pipeline + `FarePP`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

def add_features(df):
    df = df.copy()
    df["Title"] = df.Name.str.extract(r",\s*([^.]+)\.")[0].str.strip().replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["Title"] = df.Title.where(df.Title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    df["FamilySize"] = df.SibSp + df.Parch + 1
    df["IsAlone"] = (df.FamilySize == 1).astype(int)
    df["HasCabin"] = df.Cabin.notna().astype(int)
    return df

train_fe = add_features(train)
test_fe = add_features(test)

age_by_title = train_fe.groupby("Title").Age.median()
fare_by_class = train_fe.groupby("Pclass").Fare.median()
for df in (train_fe, test_fe):
    df["Age"] = df.Age.fillna(df.Title.map(age_by_title))
    df["Fare"] = df.Fare.fillna(df.Pclass.map(fare_by_class))
    df["Embarked"] = df.Embarked.fillna("S")
    df["FarePP"] = df.Fare / df.FamilySize          # the v4 addition

NUM = ["Age", "Fare", "FamilySize", "FarePP"]
CAT = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]
X, y = train_fe[NUM + CAT], train_fe.Survived

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, m in models.items():
    s = cross_val_score(Pipeline([("pre", pre), ("m", m)]), X, y, cv=cv)
    results[name] = s.mean()
    print(f"{name:22s} {s.mean():.4f} ± {s.std():.4f}")

Logistic regression    0.8339 ± 0.0107


Random forest          0.8328 ± 0.0099


Gradient boosting      0.8473 ± 0.0106


In [2]:
best = max(results, key=results.get)
print("Best model:", best, f"({results[best]:.4f})")

final = Pipeline([("pre", pre), ("m", models[best])])
final.fit(X, y)
preds = final.predict(test_fe[NUM + CAT])

sub = pd.DataFrame({"PassengerId": test_fe.PassengerId, "Survived": preds})
sub.to_csv("submission_v4.csv", index=False)
print(f"submission_v4.csv written — predicted survival rate: {preds.mean():.3f}")

Best model: Gradient boosting (0.8473)


submission_v4.csv written — predicted survival rate: 0.368


## Version history

| Version | Change | Best CV | Leaderboard |
|---|---|---|---|
| v1 | Full pipeline | 0.844 | 0.76794 |
| v2 | Lean features + rows deleted | 0.824 | 0.75837 |
| v3 | v1 minus Embarked | 0.835 | 0.76315 |
| v4 | v1 + Fare per person | (above) | — |
